# Dynamic Weighted MIA Attack

In [ ]:
import os
import pandas as pd
import numpy as np
import math
import ast
import random
import matplotlib.pyplot as plt
from transformers import pipeline
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score, precision_score, recall_score
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# Environment and Path Configuration
BASE_DIR = r"CAHNGE YOUR Path" 
SAVE_PATH = r"CAHNGE YOUR Path"
TARGET_MODELS = ['Llama2', 'Llama3', 'Qwen2.5', 'Qwen3']
MODEL_FOLDER_MAP = {'Llama2': 'Llama2', 'Llama3': 'Llama3', 'Qwen2.5': 'qwen2.5', 'Qwen3': 'qwen3'}
FILE_NAME = "inference_with_confidence_analyzed.csv"

# Hyperparameter: eta range for medical vs general feature balancing
DESTINY_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

class ESMD_Runner:
    def __init__(self):
        print("Initializing Biomedical NER engine...")
        self.ner_pipe = pipeline("ner", model="d4data/biomedical-ner-all", 
                                 aggregation_strategy="simple", device=-1)

    def extract_enhanced_mirrored_features(self, row):
        """E-SMD Engine: Extracts medical-weighted and mirrored general log-probabilities"""
        try:
            text = row['prediction']
            raw_data = ast.literal_eval(row['token_probabilities'])
            if not raw_data: return None
        except: return None

        ner_res = self.ner_pipe(text)
        med_tokens_info, gen_logps = [], []
        current_pos = 0
        
        for token_str, prob in raw_data:
            if prob <= 0: continue
            clean_t = token_str.replace('Ġ', '').replace(' ', '')
            if len(clean_t) < 2: continue 
            
            start_idx = text.find(clean_t, current_pos)
            is_med = False
            if start_idx != -1:
                end_idx = start_idx + len(clean_t)
                current_pos = end_idx
                for ent in ner_res:
                    if not (end_idx <= ent['start'] or start_idx >= ent['end']):
                        is_med = True
                        break
            
            lp = math.log(prob)
            if is_med:
                # Privacy Anchor Enhancement: Amplify signals using prediction certainty (This is an example)
                weight = 1.0 + prob * 2.0 
                med_tokens_info.append((lp, weight))
            else:
                gen_logps.append(lp)

        if not med_tokens_info: return None

        # Balanced Fragmented Sampling to break template bias
        m_count = len(med_tokens_info)
        if len(gen_logps) >= m_count:
            mirrored_gen = random.sample(gen_logps, m_count)
        else:
            mirrored_gen = gen_logps

        sum_med_lp = sum([lp * w for lp, w in med_tokens_info])
        sum_med_w = sum([w for lp, w in med_tokens_info])
        
        return {
            's_med': sum_med_lp / sum_med_w,
            's_gen': np.mean(mirrored_gen) if mirrored_gen else -18.0
        }

runner = ESMD_Runner()
all_metrics_list = []
plot_data = {m: [] for m in TARGET_MODELS}

for model_name in TARGET_MODELS:
    folder = MODEL_FOLDER_MAP.get(model_name, model_name)
    path = os.path.join(BASE_DIR, folder, FILE_NAME)
    if not os.path.exists(path): continue
    
    print(f"Analyzing Model (E-SMD High-Gain Mode): {model_name}")
    df = pd.read_csv(path).dropna(subset=['split', 'token_probabilities']).copy()
    df['label'] = df['split'].map({'trn': 1, 'tst': 0, 'test': 0})
    
    tqdm.pandas(desc=f"Extracting {model_name} privacy anchors")
    df['esmd_info'] = df.progress_apply(runner.extract_enhanced_mirrored_features, axis=1)
    df = df.dropna(subset=['esmd_info'])

    for d in DESTINY_VALUES:
        # Score Fusion: Linear combination of medical and general scores
        df['Attack_Score'] = df['esmd_info'].apply(lambda x: d * x['s_med'] + (1.0 - d) * x['s_gen'])
        
        y_true, y_scores = df['label'].values, df['Attack_Score'].values
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        
        # Search for optimal threshold based on Accuracy
        best_acc = 0
        best_metrics = {}
        thresholds = np.linspace(np.min(y_scores), np.max(y_scores), 150)
        
        for tau in thresholds:
            y_pred = (y_scores >= tau).astype(int)
            acc = accuracy_score(y_true, y_pred)
            if acc >= best_acc:
                best_acc = acc
                best_metrics = {
                    'Model': model_name,
                    'Destiny': d,
                    'AUC': round(roc_auc, 4),
                    'Accuracy': round(acc, 4),
                    'F1-score': round(f1_score(y_true, y_pred), 4),
                    'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
                    'Recall': round(recall_score(y_true, y_pred), 4)
                }
        
        all_metrics_list.append(best_metrics)
        plot_data[model_name].append(best_metrics['AUC'])

# Data Export
full_report_df = pd.DataFrame(all_metrics_list)
csv_save_name = os.path.join(SAVE_PATH, "MIA_ESMD_Complete_Metrics.csv")
full_report_df.to_csv(csv_save_name, index=False)

# Publication-Quality Plotting
plt.rcParams['font.family'] = 'serif'
plt.figure(figsize=(12, 8), dpi=300)

for m, aucs in plot_data.items():
    if aucs:
        plt.plot(DESTINY_VALUES, aucs, marker='o', label=f"{m} (AUC)", lw=3, markersize=8)

plt.title("E-SMD MIA: Privacy Leakage Evolution", fontsize=22, fontweight='bold', pad=20)
plt.xlabel("Medical Destiny ($\eta$)", fontsize=18, labelpad=10)
plt.ylabel("Attack Success Rate (AUC)", fontsize=18, labelpad=10)
plt.xticks(DESTINY_VALUES, fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14, loc='lower right', frameon=True, shadow=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig(os.path.join(SAVE_PATH, "MIA_ESMD_Trend_Analysis.png"))
plt.show()

print(f"Experimental report saved to: {csv_save_name}")

### AUC curve

In [ ]:
import pandas as pd
import numpy as np
import json
import math
import os
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# Configuration and Paths
BASE_DIR = r"CAHNGE YOUR Path"
SAVE_PATH = r"CAHNGE YOUR Path"
TARGET_MODELS = ['Llama2', 'Llama3', 'Qwen2.5', 'Qwen3']
MODEL_FOLDER_MAP = {'Llama2': 'Llama2', 'Llama3': 'Llama3', 'Qwen2.5': 'qwen2.5', 'Qwen3': 'qwen3'}
FILE_NAME = "inference_with_confidence_analyzed.csv"

# Hyperparameters example
DESTINY = 0.9 
K_RATIO = 0.1 
MEDICAL_FACTOR = 4.0 
GENERAL_FACTOR = 0.1 

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# Initialize Medical NER Engine
print("Initializing Medical NER engine...")
ner_pipe = pipeline("ner", model="d4data/biomedical-ner-all", 
                    aggregation_strategy="simple", device=-1)

def calculate_medical_weighted_features(row, destiny):
    """Calculate weighted token probability features based on medical attributes"""
    default_vals = (-10.0, 0.0, 0.0) 
    
    try:
        text = row['prediction']
        prob_str = row['token_probabilities']
        if pd.isna(prob_str): return default_vals
        token_probs = json.loads(prob_str.replace("'", '"'))
        if not token_probs: return default_vals
    except:
        return default_vals

    # Identify medical entity spans
    ner_res = ner_pipe(text)
    med_spans = [(e['start'], e['end']) for e in ner_res]

    weighted_log_probs = []
    current_pos = 0
    
    for token_str, prob in token_probs:
        if prob <= 0: continue
        
        # Positional alignment
        clean_t = token_str.replace('Ġ', '').replace(' ', '')
        start_idx = text.find(clean_t, current_pos)
        
        is_med = False
        if start_idx != -1:
            end_idx = start_idx + len(clean_t)
            current_pos = end_idx
            is_med = any(not (end_idx <= ms[0] or start_idx >= ms[1]) for ms in med_spans)
        
        # Apply dynamic weighting
        weight = (1 - destiny) * 1.0 + destiny * (MEDICAL_FACTOR if is_med else GENERAL_FACTOR)
        weighted_log_probs.append(math.log(prob) * weight)

    if not weighted_log_probs: return default_vals

    # Feature extraction
    sorted_weighted = sorted(weighted_log_probs)
    num_k = max(1, int(len(sorted_weighted) * K_RATIO))
    weighted_min_k = np.mean(sorted_weighted[:num_k])
    
    # Statistical exposure for low probability distribution
    med_exposure = np.sum([w for w in weighted_log_probs if w < -5]) 
    
    return weighted_min_k, med_exposure, len(weighted_log_probs)

# Execution Loop
all_model_metrics = []
plt.figure(figsize=(10, 8))

for model_name in TARGET_MODELS:
    folder = MODEL_FOLDER_MAP.get(model_name, model_name)
    path = os.path.join(BASE_DIR, folder, FILE_NAME)
    
    if not os.path.exists(path):
        print(f"Skipping {model_name}: File not found at {path}")
        continue
    
    print(f"Executing Medical-Weighted MIA attack on {model_name}...")
    df = pd.read_csv(path)
    
    tqdm.pandas(desc=f"Extracting features: {model_name}")
    features_res = df.progress_apply(lambda r: calculate_medical_weighted_features(r, DESTINY), axis=1)
    df[['W_Min_K', 'Med_Exp', 'Token_Len']] = pd.DataFrame(features_res.tolist(), index=df.index)
    
    # Mapping labels: trn -> 1 (Member), test -> 0 (Non-member)
    feature_cols = ['W_Min_K', 'Med_Exp', 'Overall_Avg_Conf', 'Perplexity']
    data = df.dropna(subset=feature_cols + ['split']).copy()
    data['label'] = data['split'].map({'trn': 1, 'tst': 0, 'test': 0})
    
    X = data[feature_cols].values
    y = data['label'].values
    
    # Stratified 5-Fold Cross-Validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_probs_all = np.zeros(len(y))
    y_pred_all = np.zeros(len(y))
    
    for tr_idx, te_idx in skf.split(X, y):
        clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
        clf.fit(X[tr_idx], y[tr_idx])
        y_probs_all[te_idx] = clf.predict_proba(X[te_idx])[:, 1]
        y_pred_all[te_idx] = clf.predict(X[te_idx])
    
    # Evaluation Metrics
    fpr, tpr, _ = roc_curve(y, y_probs_all)
    roc_auc = auc(fpr, tpr)
    
    all_model_metrics.append({
        'Model': model_name,
        'AUC': round(roc_auc, 4),
        'F1-Score': round(f1_score(y, y_pred_all), 4),
        'Precision': round(precision_score(y, y_pred_all), 4),
        'Recall': round(recall_score(y, y_pred_all), 4),
        'Accuracy': round(accuracy_score(y, y_pred_all), 4)
    })
    
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc:.4f})', lw=2)
    
    # Save individual attack scores
    data['attack_prob'] = y_probs_all
    data.to_csv(os.path.join(SAVE_PATH, f"MIA_Attack_Score_{model_name}.csv"), index=False)

# Results and Visualization
metrics_df = pd.DataFrame(all_model_metrics)
print("\nFinal MIA Attack Summary")
print(f"Hyperparameters: Destiny={DESTINY}")
print("-" * 30)
print(metrics_df.to_string(index=False))

# Export final report
final_report_path = os.path.join(SAVE_PATH, "MIA_Final_Comparison_Report.csv")
metrics_df.to_csv(final_report_path, index=False)
print(f"Summary report saved to: {final_report_path}")

# Plot styling
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.title(f"MIA Attack ROC: Medical-Weighted Fusion (Destiny={DESTINY})")
plt.xlabel("False Positive Rate (Non-members)")
plt.ylabel("True Positive Rate (Members)")
plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.show()